# Placement and protection pilot v2

Local CPU simulation; no Ray, GPU, cloud, Z3 or learning training. Read `README.md` and `MODEL.md` for assumptions.

The objective remains **completion time including failures**, subject to cost and deadline requirements. V2 tests whether accounting for failures improves decisions beyond speed-based grouping. The failure-aware and failure-unaware searches score the **same candidate pool**, then freeze their choices before held-out evaluation.

Restart the kernel after pulling updated Python code. Run setup/settings, inspect the cases, then **Run / resume**. Each invocation pauses after 30 minutes; rerun the same cell to continue. Plotting cells only read saved files. Use a new run name whenever source/config changes. Existing `pilot_v1` output is preserved.


In [ ]:
from pathlib import Path
import json
import sys

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "learnedplacement" / "runner.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LearnedPlacement checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from learnedplacement.runner import run, status, read_json, paired_report, export_run
from learnedplacement.model import make_cases
print("Repository:", ROOT)
print("Python:", sys.version.split()[0])


## Editable settings

48 cases = 2 sizes × 4 regimes × 2 instances × 3 profiles. Seven methods, each evaluated on 256 paired held-out traces (86,016 evaluations). Failure-aware scoring uses 32 separate planning traces; the deterministic failure-free objective needs one simulation per candidate.

Both matched searches consider at most 64 identical candidates, with a shared 30-second planning cap per case. A partial candidate is discarded for both. This matches candidate choices, not CPU effort. The candidate generator sees neither failure rates nor scored outcomes.

Profiles: original reference settings; tighter budgets; and a combined recovery stress case. These are synthetic sensitivity assumptions, not measured platform parameters. Inspect screen failures as outcomes, rather than changing settings until a preferred method wins. `MAX_MINUTES` alone can change on resume.


In [ ]:
RUN_NAME = "pilot_v2"
MAX_MINUTES = 30

CONFIG = read_json(ROOT / "configs" / "pilot_v2.json")
CONFIG["active_nodes"] = [8, 16]
CONFIG["regimes"] = ["homogeneous", "cheap_risky", "fast_risky", "correlated"]
CONFIG["instances_per_regime"] = 2
CONFIG["planning_traces"] = 32
CONFIG["evaluation_traces"] = 256
CONFIG["matched_candidates"] = 64
CONFIG["matched_seconds"] = 30.0
CONFIG["workload"]["backup_budget_gib"] = 8.0
CONFIG["workload"]["backup_every"] = 20
CONFIG["workload"]["checkpoint_every"] = 80
CONFIG["workload"]["required_success"] = 0.95
# Profiles and all their overrides are printed below and editable in CONFIG.
# To run only the reference profile, before starting a NEW run:
# CONFIG["profiles"] = CONFIG["profiles"][:1]

OUTPUT = ROOT / "results" / RUN_NAME
print("Output:", OUTPUT)
print(json.dumps(CONFIG, indent=2))


## Inspect one generated case

No simulation runs in this cell. Machine prices, speeds, risk and bandwidth are generated in `learnedplacement/model.py`. Here you can see the actual deadlines and cost budgets before starting.


In [ ]:
cases = make_cases(CONFIG)  # Input generation only; no simulations.
print("Cases:", len(cases), "Methods:", len(CONFIG["methods"]))
print("Held-out evaluations:", len(cases) * len(CONFIG["methods"]) * CONFIG["evaluation_traces"])
for example in cases[:len(CONFIG["profiles"])]:
    print(example.case_id,
          "deadline:", round(example.deadline_s, 2),
          "cost budget:", round(example.cost_budget_usd, 4),
          "storage GiB/s:", example.storage_gib_s,
          "replacement s:", example.replacement_s)
print(json.dumps([vars(n) for n in cases[0].nodes], indent=2))


## Run / resume

This cell starts the experiment. It uses one CPU process. Completed cases/plans and batches of observations save locally. Interrupting the cell is supported. A killed kernel may leave `.run.lock`; remove that file only after confirming the old process has stopped.

The exact source/configuration fingerprint is checked on resume. If you pull changed core code or edit the configuration, use a new run name. No completed result is overwritten by a different experiment.


In [ ]:
run_status = run(config=CONFIG, output=OUTPUT, max_minutes=MAX_MINUTES)
print(json.dumps(run_status, indent=2))


## Progress and results table

You can run these cells after a pause. They only read saved results. `statistical_screen_passed` is a per-plan Monte Carlo screen, not a real-world guarantee. Censored traces have no reported mean completion time or total cost.


In [ ]:
print(json.dumps(status(OUTPUT), indent=2))

import pandas as pd
from IPython.display import display, FileLink

summary_path = OUTPUT / "summary.csv"
if summary_path.exists():
    results = pd.read_csv(summary_path)
    display(results[["case_id", "profile", "method", "mean_completion_s",
                     "failure_free_completion_s", "failure_penalty_s", "mean_cost_usd",
                     "deadline_success", "deadline_success_lower95",
                     "statistical_screen_passed", "censored_traces", "planning_s", "candidates"]])
else:
    print("No completed comparisons yet. Resume the run above.")


## Plots from saved results

Choose the size/profile to avoid pooling different assumptions. Lower time at higher cost is a trade-off; inspect feasibility. No-backup ablations are omitted from latency plots because they inherit their parent's search.

The diagnostic plot decomposes **total difference = failure-free difference + failure-penalty difference**. Failure penalty includes rework, reduced throughput, detection, waiting, and changes in checkpoint/backup overhead, not just time labeled recovery. Only uncensored pairs can have this decomposition; the results table retains censored cases. There are no vertical bars or bottom captions.


In [ ]:
from learnedplacement.plots import plot_results, plot_failure_effects

ACTIVE_NODES_TO_PLOT = 8  # Or 16.
PROFILE_TO_PLOT = "reference"  # Or "tight", "recovery_stress".
table = plot_results(OUTPUT, active_nodes=ACTIVE_NODES_TO_PLOT, profile=PROFILE_TO_PLOT)
if table is not None:
    display(table)
diagnostics = plot_failure_effects(OUTPUT, active_nodes=ACTIVE_NODES_TO_PLOT,
                                   profile=PROFILE_TO_PLOT)
if diagnostics is not None:
    display(diagnostics)


## Paired comparisons

Negative differences favor the selected planner. The first comparison isolates failure-aware scoring on a shared candidate pool. The second tests against speed grouping (which also gets a failure-aware backup toggle). The third isolates additional backups at exactly the same placement.

Check both feasibility screens, actual candidate counts, and planning latency. Zero completed search candidates means a fallback, not an optimized result. Bootstrap intervals capture Monte Carlo variability within each synthetic case, not model uncertainty or generalization. Do not reselect plans using these evaluation results.


In [ ]:
for baseline in ["matched_failure_unaware", "speed_grouped", "matched_no_backup"]:
    print("matched_failure_aware vs", baseline)
    comparisons = paired_report(OUTPUT, selected="matched_failure_aware", baseline=baseline)
    display(pd.DataFrame(comparisons))


## Compact export for sharing

Stop or pause the runner first. The default export contains only analysis data: configuration/source hashes, case inputs, selected plans, summaries, failure-free measurements and paired statistics for every available method pair. It automatically computes the comparisons from saved observations; running the plotting/comparison cells first is optional.

Detailed traces, full candidate-search logs, plots and source copies stay local. Nothing is uploaded automatically. This export reads saved results and does not run simulations. It works for incomplete runs too, with shared-prefix statistics labeled as partial. Approximate paired mean intervals here use a normal approximation, not the bootstrap used in the comparison cell.

After pulling this export update, restart the kernel and run only setup, settings and this export cell to share an existing completed run. Do not rerun the experiment. A full troubleshooting archive is optional: `export_run(OUTPUT, full=True)`.


In [ ]:
from IPython.display import display, FileLink
archive = export_run(OUTPUT)  # Compact analysis export by default.
print("Share this ZIP:", archive)
print("Size:", round(archive.stat().st_size / 1024, 1), "KiB")
import os
display(FileLink(os.path.relpath(archive, Path.cwd())))
